In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# BigQuery DataFrames - Local Peek Cache (Experimental)

This notebook introduces and demonstrates the experimental **local peek cache** feature in BigQuery DataFrames (`bigframes`).

## Overview

When doing interactive data science in Jupyter notebooks, a common workflow is to perform a transformation, look at the first few rows (using `.head()`, `.peek()`, or by simply displaying the DataFrame), and then perform another transformation.

By default, BigQuery DataFrames translates each of these steps into a BigQuery query. Even if you are only looking at the first 10 rows, BigQuery might scan gigabytes or terabytes of data to compute the results. This results in:
1. **High query latency** (each cell execution takes a few seconds).
2. **High query costs** (each query incurs scanning costs).

The **Local Peek Cache** addresses this. When enabled:
* The first time you peek at a DataFrame, BigQuery DataFrames pulls a larger local sample (e.g. 10,000 rows) and caches it in memory.
* Subsequent compatible operations (e.g. column projections, filters, math operations, custom UDFs) on that DataFrame that also request a peek will be computed **locally** (via Polars) on the cached sample.
* This completely avoids BigQuery execution for intermediate checks, making your interactive loop feel **instantaneous** and **cost-free**.

---

### In this notebook, you will learn:
1. How to enable the peek cache using `bpd.options.compute.enable_peek_cache`.
2. How the peek cache speeds up interactive queries over a large public dataset.
3. How to check execution times between cache hits (local Polars execution) and cache misses (BigQuery execution).


## Setup & Import Libraries


In [2]:
import time
import bigframes.pandas as bpd

# Configure the BigQuery project to 'bigframes-dev' for this demo
bpd.options.bigquery.project = "bigframes-dev"
bpd.options.bigquery.ordering_mode = 'partial'


# Enable the Local Peek Cache
bpd.options.compute.enable_peek_cache = True
# Set the size of the cached local sample
bpd.options.compute.peek_cache_size = 10000

# Initialize global session
session = bpd.get_global_session()
print(f"Using Google Cloud Project: {session.bqclient.project}")
print(f"Peek cache enabled: {bpd.options.compute.enable_peek_cache}")
print(f"Peek cache size: {bpd.options.compute.peek_cache_size}")


/usr/local/google/home/tbergeron/src/google-cloud-python/packages/bigframes/bigframes/core/global_session.py:113: DefaultLocationWarning: No explicit location is set, so using location US for the session.
  _global_session = bigframes.session.connect(


Using Google Cloud Project: bigframes-dev
Peek cache enabled: True
Peek cache size: 10000


## Loading a Large Dataset

We will read from `bigquery-public-data.chicago_taxi_trips.taxi_trips`, which contains over 200 million rows. 

First, let's create a DataFrame representing the table. Since BigQuery DataFrames is lazy by default, this operation is instantaneous and doesn't execute a query yet.


In [3]:
df = bpd.read_gbq("bigquery-public-data.chicago_taxi_trips.taxi_trips")
print("DataFrame created lazily.")


DataFrame created lazily.


## Step 1: The Initial Peek (Cache Miss)

Let's peek at the first 10 rows of our DataFrame. 

Under the hood:
1. Since the peek cache is currently empty, this will result in a **cache miss**.
2. BigQuery DataFrames will issue a query to BigQuery to fetch the first **10,000 rows** (instead of just 10).
3. The 10,000 rows are downloaded and cached locally as a PyArrow table.
4. Finally, the first 10 rows are displayed.

Because this runs a real BigQuery query and downloads data, it will take a few seconds. Let's measure the time:


In [4]:
start_time = time.time()
# Display the preview
print(df.peek(10))
end_time = time.time()
print(f"\nInitial peek execution time: {end_time - start_time:.4f} seconds")


                                 unique_key  \
0  b99416692f5bc01cdb444cbcec7e1105d30d455d   
1  b99887012c8beccc1dae3f6fcde2fc94ff0ef7b1   
2  b98b6b7afe1f8065d8ba71cd023d9a13f2460f84   
3  b9865af3673441b1a79973ad08bd9909bd422af4   
4  e04dbd9a8ba723cc785c4b0398106b72e16d7870   
5  b997997e971c079dd0986102def2de49e879efe5   
6  b989b45f0f49b64a8cc59f85aa2b49604286755d   
7  b9929c08951468b8199c0804456e11ad988c5792   
8  b987f48069fbdffcdbb9cfcc24f67d7814a03645   
9  b9958c55b8d387cf335e4bb7703aeca2592fb847   

                                             taxi_id  \
0  8dd990653cc2793d96b80ae45e928afc9a590b21b491ee...   
1  d156d4adff01addd62b65753ce67dc8e04bdaf60f7eb14...   
2  beefd3462e3f5a8e854942a2796876f6db73ebbd25b435...   
3  ef249ea2c31ccb20ce9eb258d3116254a6f9fe05c7cc84...   
4  45002471b85d61e62cf602a15145d2bff912e080f50470...   
5  fa0a84435b292833f8429400e94600f4b71dd17cafbaf5...   
6  8423aff4ae55153cc18d090c35b73b5cb6376b828d836d...   
7  9b2a23587b25fe43998076b3761b979

## Step 2: Interactive Exploration (Cache Hits)

Now that we have the first 10,000 rows cached locally, subsequent interactive transformations and peeks will execute **locally on the cached sample** using Polars.

Let's perform a filter and column projection:
1. Filter trips that cost more than $20 (`fare > 20`).
2. Compute the total cost including tips (`trip_total = fare + tips`).
3. Display the first 10 rows of the result.

This will execute completely locally without querying BigQuery, which means it should take **milliseconds** instead of seconds! Let's measure the time:


In [5]:
start_time = time.time()

# 1. Filter trips where fare is greater than 20
df_filtered = df[df["fare"] > 20]

# 2. Compute a new column
df_filtered = df_filtered.assign(
    trip_total_calculated = df_filtered["fare"] + df_filtered["tips"]
)

# 3. Select a subset of columns
df_select = df_filtered[["unique_key", "fare", "tips", "trip_total_calculated"]]

# 4. Display/peek the first 10 rows (executed locally on the cache!)
preview_local = df_select.peek(10)
print(preview_local)

end_time = time.time()
print(f"\nSubsequent local peek execution time: {end_time - start_time:.4f} seconds")


                                 unique_key   fare   tips  \
0  e04dbd9a8ba723cc785c4b0398106b72e16d7870  24.05   6.01   
1  b9929c08951468b8199c0804456e11ad988c5792   44.0    8.9   
2  b987f48069fbdffcdbb9cfcc24f67d7814a03645  45.75  12.65   
3  b98db59faf6d2817996bfd570293905569b2d6df  44.75   7.39   
4  b987f83bfac967768d652b23b2ad4241dbf6f393  32.75   7.65   
5  b9add5cbe221484e03e021e4120aca4cd59ea137   24.0    5.3   
6  b99d19ad1b5b3073f4497c340d305ddbc62f46d4   45.0    0.0   
7  355a1f57006cb193c8ff9b8bb010703b298b98f7  37.25   7.45   
8  b99dac71abd5e891583c1a3eab96570a86965c05   51.1   12.9   
9  b99cd6ca1bb73117e5ed0aa9374bdbc746b451ab  45.25   9.95   

   trip_total_calculated  
0                  30.06  
1                   52.9  
2                   58.4  
3                  52.14  
4                   40.4  
5                   29.3  
6                   45.0  
7                   44.7  
8                   64.0  
9                   55.2  

Subsequent local peek executio

## Step 3: Full Execution (Bypassing the Cache)

The local peek cache is only used when you are *peeking* at a subset of rows (e.g. using `head()`). 

If you perform an operation that requires processing the entire dataset (such as `.to_pandas()`, `.to_gbq()`, or computing the global `.mean()`), BigQuery DataFrames automatically falls back to full BigQuery execution to ensure the results are computed correctly over the entire 200M+ rows.

Let's look at the execution flow of computing the average fare over the entire dataset:


In [6]:
start_time = time.time()

# Compute the mean fare over the entire dataset (200M+ rows)
# This will execute on BigQuery (bypassing the local cache to ensure correctness)
mean_fare = df["fare"].mean()
print(f"Average fare: ${mean_fare:.2f}")

end_time = time.time()
print(f"Full BigQuery query execution time: {end_time - start_time:.4f} seconds")


Average fare: $13.82
Full BigQuery query execution time: 1.0945 seconds


## Step 4: Aggregation and Complete Cache Exploration

A particularly powerful workflow with the Local Peek Cache is **aggregation followed by rapid exploration**.

If you run an aggregation (like `groupby.agg`) that reduces a massive dataset to a small number of rows (e.g., less than 10,000 rows):
1. Calling a full execution method like `.to_pandas()` or displaying the aggregated DataFrame will query BigQuery once to compute the aggregated data.
2. Because the total number of rows in the aggregated result is smaller than the `peek_cache_size`, the entire result fits in the cache.
3. BigQuery DataFrames caches the result and marks the cache entry as **complete** (`is_complete=True`).
4. **Any subsequent operation** on this aggregated DataFrame (filtering, sorting, column math, etc.) will run **100% locally** using Polars, even if you request full execution (like `.to_pandas()`), without ever querying BigQuery again!

Let's start by calculating the average fare, average tips, and total trip count grouped by payment type over the 200M+ taxi trips table.


In [7]:
start_time = time.time()

# 1. Group by payment_type and compute stats
df_agg = df.groupby("payment_type").agg(
    fare_mean=("fare", "mean"),
    trip_count=("fare", "count"),
    tips_mean=("tips", "mean")
).reset_index()

# 2. Perform a full download to pandas (initial compilation and BQ query)
print("Running BigQuery aggregation query...")
df_agg_pd = df_agg.to_pandas()
print(df_agg_pd)

end_time = time.time()
print(f"\nAggregation query and download execution time: {end_time - start_time:.4f} seconds")


Running BigQuery aggregation query...


   payment_type  fare_mean  trip_count  tips_mean
0          Cash  11.639907   119532408   0.005749
1   Credit Card  16.430913    86256408   3.573094
2       Dispute  13.940387       88949   0.009501
3        Mobile  15.972737     2602938   3.161899
4     No Charge  14.576718      826657    0.70816
5         Pcard   9.502388       36895   0.232494
6        Prcard  22.824426     2202147   0.193083
7       Prepaid  22.035569        1898   0.007782
8         Split   14.39493        3442   3.201069
9       Unknown  19.505744     1538372   0.088709
10     Way2ride  12.892676         142   2.411479

Aggregation query and download execution time: 1.3534 seconds


### Downstream Local Analysis (Cache Hit)

Since the output of the aggregation is small and has been fully downloaded, it is stored in the cache as a **complete** entry. 

Let's perform further exploration on this aggregated data:
1. Filter out payment types with less than 1,000 trips.
2. Calculate the tip percentage (`tip_percentage = (tips_mean / fare_mean) * 100`).
3. Sort by `tip_percentage` descending.
4. Download and display the full table using `.to_pandas()`.

This entire block runs on the local engine (Polars) over the cached Arrow table in **milliseconds**, with zero BigQuery charges!


In [8]:
start_time = time.time()

# 1. Filter out rare payment types (less than 1,000 trips)
df_common = df_agg[df_agg["trip_count"] >= 1000]

# 2. Compute the tip percentage
df_common = df_common.assign(
    tip_percentage = (df_common["tips_mean"] / df_common["fare_mean"]) * 100
)

# 3. Sort by tip percentage descending
df_sorted = df_common.sort_values(by="tip_percentage", ascending=False)

# 4. Perform a full download to pandas (runs 100% locally on the complete cache!)
result_pd = df_sorted.to_pandas()
print(result_pd)

end_time = time.time()
print(f"\nDownstream local exploration execution time: {end_time - start_time:.4f} seconds")


  payment_type  fare_mean  trip_count  tips_mean  tip_percentage
0        Split   14.39493        3442   3.201069       22.237476
1  Credit Card  16.430913    86256408   3.573094       21.746168
2       Mobile  15.972737     2602938   3.161899       19.795601
3    No Charge  14.576718      826657    0.70816        4.858156
4        Pcard   9.502388       36895   0.232494        2.446688
5       Prcard  22.824426     2202147   0.193083        0.845949
6      Unknown  19.505744     1538372   0.088709        0.454783
7      Dispute  13.940387       88949   0.009501        0.068154
8         Cash  11.639907   119532408   0.005749        0.049386
9      Prepaid  22.035569        1898   0.007782        0.035315

Downstream local exploration execution time: 0.0223 seconds
